# 🎟️ Targeted YuedPao Coupon & Discount Code Ticket Scraper
**Chatbot YuedPao - Coupon Ticket Scraper, ChromaDB Vector Indexing & LINE Flex Message Generator**

สมุดโน้ตสำหรับสกัดตั๋วคูปองส่วนลดจากเว็บ YuedPao และทำการโหลดเข้าทั้ง **SQLite DB** และ **ChromaDB Vector Store** แบบครบวงจร:
1. **Scrape Coupon Container (`class="w-full flex"`)** $\rightarrow$ ดึงตั๋วคูปองทั้งหมดจากป๊อปอัปคูปอง
2. **Auto-Click Terms Modal (`เงื่อนไขการใช้งาน`)** $\rightarrow$ สกัด **รหัสคูปอง (Code e.g. `NEWMEMBER5`, `YUEDPAO006`)**, **ระยะเวลาใช้งาน (Valid Duration)**, และ **เงื่อนไขเพิ่มเติม (Detailed Condition)**
3. **Save to SQLite (`coupons` Table)** $\rightarrow$ บันทึกลงตาราง `coupons` ใน `yuedpao_chatbot.db` (Persistence Layer)
4. **Index to ChromaDB Vector DB (`yuedpao_coupons_e5`) & BM25** $\rightarrow$ โหลดข้อมูลเข้า ChromaDB เวกเตอร์สโตร์และสร้างดัชนี BM25 สำหรับ RRF Hybrid Search
5. **Test RRF Hybrid Search** $\rightarrow$ ทดสอบค้นหาคูปองด้วยคำถามภาษาธรรมชาติ (Natural Language Query)
6. **LINE Flex Message Generator** $\rightarrow$ แปลงข้อมูลคูปองเป็นโครงสร้าง **LINE Flex Message (Carousel)** พร้อมปุ่มคัดลอกรหัสโค้ด

In [ ]:
# ติดตั้งและตั้งค่าไลบรารีที่จำเป็น
import os
import sys
import re
import time
import json
import sqlite3
import numpy as np
import pandas as pd
from typing import List, Dict, Any, Optional
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By

try:
    import chromadb
    from sentence_transformers import SentenceTransformer
    from rank_bm25 import BM25Okapi
    from pythainlp.tokenize import word_tokenize
except ImportError as e:
    print(f"⚠️ Warning: {e}")

if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')
print("✅ โหลดโมเดลและไลบรารีสำหรับ Coupon & ChromaDB Indexer เรียบร้อย!")

## 🌐 Step 1: สกัดตั๋วคูปอง `w-full flex` พร้อมกดป๊อปอัป `เงื่อนไขการใช้งาน` ดึงโค้ด & ระยะเวลา

In [ ]:
def init_headless_driver() -> webdriver.Chrome:
    chrome_options = Options()
    chrome_options.add_argument("--headless")
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
    return webdriver.Chrome(options=chrome_options)

def scrape_yuedpao_coupons_with_terms() -> List[Dict[str, Any]]:
    driver = init_headless_driver()
    url = "https://www.yuedpao.com"
    print(f"⏳ กำลังเปิดหน้าเว็บ {url}...")
    
    coupons = []
    seen_keys = set()
    
    try:
        driver.get(url)
        time.sleep(5)
        
        # 1. ยอมรับคุกกี้เพื่อป้องกัน Cookie Overlay บังการคลิก
        driver.execute_script("""
            let acceptBtn = Array.from(document.querySelectorAll('button')).find(b => b.textContent.includes('ยอมรับทั้งหมด'));
            if (acceptBtn) acceptBtn.click();
        """)
        time.sleep(2)
        
        # 2. คลิกปุ่ม 'คูปองส่วนลด'
        chips = driver.find_elements(By.XPATH, "//*[contains(text(), 'คูปองส่วนลด')]")
        if chips:
            print("👆 คลิกปุ่ม 'คูปองส่วนลด' เพื่อเปิดรายการตั๋วคูปอง...")
            driver.execute_script("arguments[0].click();", chips[0])
            time.sleep(4)
            
        # 3. ค้นหาตั๋วคูปอง container class="w-full flex"
        soup = BeautifulSoup(driver.page_source, "html.parser")
        cards = soup.find_all(class_=lambda c: c and 'w-full' in c.split() and 'flex' in c.split())
        
        print(f"📦 พบ Container คูปองทั้งหมด {len(cards)} โหนด -> กำลังสกัดข้อมูลและกดดูเงื่อนไข...")
        terms_buttons = driver.find_elements(By.XPATH, "//p[contains(text(), 'เงื่อนไขการใช้งาน')] | //*[text()='เงื่อนไขการใช้งาน']")
        
        for idx, card in enumerate(cards):
            text_content = card.get_text(separator=" ", strip=True)
            if not any(kw in text_content for kw in ["ส่วนลด", "ขั้นต่ำ", "ใช้ได้ถึง", "คูปอง"]):
                continue
                
            badge_container = card.find(class_=lambda c: c and 'flex' in c.split() and 'justify-center' in c.split() and 'items-center' in c.split() and 'flex-col' in c.split())
            badge_title = "ส่วนลดสินค้า"
            svg_html = ""
            badge_bg_color = "#EF4444"
            
            if badge_container:
                title_tag = badge_container.find(["h6", "span", "p", "div"])
                if title_tag:
                    badge_title = title_tag.get_text(strip=True)
                svg_tag = badge_container.find("svg")
                if svg_tag:
                    svg_html = str(svg_tag)
                
                badge_parent = badge_container.parent
                while badge_parent and badge_parent != card:
                    style_attr = badge_parent.get("style", "")
                    if "background" in style_attr or "color" in style_attr:
                        m = re.search(r'background(?:-color)?:\s*([^;]+)', style_attr)
                        if m:
                            badge_bg_color = m.group(1).strip()
                            break
                    badge_parent = badge_parent.parent
            
            discount_title = ""
            min_spend = "0"
            expiry_date = ""
            eligibility_tag = "ทุกสมาชิก"
            
            disc_m = re.search(r'(ส่วนลด\s*\d+\s*(?:%|บาท)?)', text_content)
            discount_title = disc_m.group(1).strip() if disc_m else badge_title
            
            spend_m = re.search(r'ขั้นต่ำ\s*([\d,]+)', text_content)
            min_spend = spend_m.group(1).replace(",", "").strip() if spend_m else "0"
            
            exp_m = re.search(r'ใช้ได้ถึง\s*([\d\w\.\s]+?)(?:ลูกค้าใหม่|เงื่อนไข|เก็บ|$)', text_content)
            expiry_date = exp_m.group(1).strip() if exp_m else ""
            
            if "ลูกค้าใหม่" in text_content:
                eligibility_tag = "ลูกค้าใหม่"
            elif "สมาชิก" in text_content:
                eligibility_tag = "สมาชิก YuedPao"
                
            unique_key = f"{discount_title}_{min_spend}_{expiry_date}"
            if unique_key in seen_keys:
                continue
            seen_keys.add(unique_key)
            
            coupon_code = f"YUEDPAO{idx+1:02d}"
            valid_duration = expiry_date
            detailed_condition = f"เมื่อซื้อสินค้า {min_spend} บาทขึ้นไป" if min_spend != "0" else "ไม่มีขั้นต่ำ"
            
            if idx < len(terms_buttons):
                try:
                    print(f"  🔍 กดเปิดเงื่อนไขการใช้งานของตั๋ว #{len(coupons)+1} ({discount_title})...")
                    driver.execute_script("arguments[0].click();", terms_buttons[idx])
                    time.sleep(2.5)
                    
                    modal_soup = BeautifulSoup(driver.page_source, "html.parser")
                    dialogs = modal_soup.find_all(class_=lambda c: c and ('MuiDialog-paper' in c or 'MuiPaper-root' in c or 'MuiPopover-paper' in c))
                    
                    for d in dialogs:
                        d_text = d.get_text(separator="\n", strip=True)
                        if any(kw in d_text for kw in ["ระยะเวลา", "เมื่อซื้อ", "โค้ด"]) and "ยอมรับทั้งหมด" not in d_text:
                            code_matches = re.findall(r'([A-Z0-9_-]{4,20})', d_text)
                            for cm in code_matches:
                                if cm not in ["MUI", "TYPOGRAPHY", "BUTTON", "FLEX", "CONTAINER"]:
                                    coupon_code = cm
                                    break
                                    
                            dur_m = re.search(r'ระยะเวลา\s*
?\s*([\d\w\s:-]+ - [\d\w\s:-]+)', d_text)
                            if dur_m:
                                valid_duration = dur_m.group(1).strip()
                                
                            cond_m = re.search(r'(เมื่อซื้อสินค้า[\d\w\s]+ขึ้นไป)', d_text)
                            if cond_m:
                                detailed_condition = cond_m.group(1).strip()
                            break
                            
                    driver.execute_script("""
                        let closeBtn = document.querySelector('.MuiDialog-root button, .MuiModal-root button, [aria-label="close"]');
                        if (closeBtn) closeBtn.click();
                        else document.querySelector('.MuiBackdrop-root')?.click();
                    """)
                    time.sleep(1)
                except Exception as e:
                    print(f"  ⚠️ ไม่สามารถเปิดป๊อปอัปได้: {e}")
            
            coupons.append({
                "coupon_id": f"COUPON_{len(coupons) + 1:03d}",
                "badge_title": badge_title,
                "badge_bg_color": badge_bg_color,
                "discount_title": discount_title,
                "coupon_code": coupon_code,
                "min_spend": int(min_spend) if min_spend.isdigit() else 0,
                "expiry_date": expiry_date,
                "valid_duration": valid_duration,
                "detailed_condition": detailed_condition,
                "eligibility_tag": eligibility_tag,
                "action_status": "เก็บ",
                "badge_svg_html": svg_html,
                "raw_container_html": str(card)
            })
            
        return coupons
    finally:
        driver.quit()

coupons_data = scrape_yuedpao_coupons_with_terms()
print(f"🎉 สกัดข้อมูลคูปองและเงื่อนไขการใช้งานสำเร็จรวม {len(coupons_data)} รายการ!")

## 📊 Step 2: แสดงผลตารางสรุปคูปอง (Pandas DataFrame Summary)

In [ ]:
df_coupons = pd.DataFrame(coupons_data)
display_cols = ["coupon_id", "badge_title", "discount_title", "coupon_code", "min_spend", "valid_duration", "detailed_condition", "eligibility_tag"]
if not df_coupons.empty:
    display(df_coupons[display_cols])
else:
    print("⚠️ ไม่พบข้อมูลคูปอง")

## 🗄️ Step 3: บันทึกข้อมูลคูปองลงตาราง `coupons` ใน SQLite Database (`yuedpao_chatbot.db`)

In [ ]:
def save_coupons_to_db(coupons: List[Dict[str, Any]], db_path: str = "../yuedpao_chatbot.db"):
    if not os.path.exists(os.path.dirname(db_path)) and os.path.dirname(db_path) != "":
        os.makedirs(os.path.dirname(db_path), exist_ok=True)
        
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    
    cursor.execute("DROP TABLE IF EXISTS coupons")
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS coupons (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        coupon_id TEXT UNIQUE,
        badge_title TEXT,
        badge_bg_color TEXT,
        discount_title TEXT,
        coupon_code TEXT,
        min_spend INTEGER,
        expiry_date TEXT,
        valid_duration TEXT,
        detailed_condition TEXT,
        eligibility_tag TEXT,
        action_status TEXT,
        badge_svg_html TEXT,
        raw_container_html TEXT,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
    """
    )
    
    for c in coupons:
        cursor.execute("""
        INSERT INTO coupons (
            coupon_id, badge_title, badge_bg_color, discount_title,
            coupon_code, min_spend, expiry_date, valid_duration,
            detailed_condition, eligibility_tag, action_status,
            badge_svg_html, raw_container_html
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, (
            c["coupon_id"], c["badge_title"], c["badge_bg_color"], c["discount_title"],
            c["coupon_code"], c["min_spend"], c["expiry_date"], c["valid_duration"],
            c["detailed_condition"], c["eligibility_tag"], c["action_status"],
            c["badge_svg_html"], c["raw_container_html"]
        ))
        
    conn.commit()
    cursor.execute("SELECT COUNT(*) FROM coupons")
    total = cursor.fetchone()[0]
    conn.close()
    print(f"💾 บันทึกคูปองลงตาราง 'coupons' ใน {db_path} สำเร็จรวม {total} รายการ!")

save_coupons_to_db(coupons_data)

## 🤖 Step 4: โหลดและแปลงคูปองเข้า ChromaDB Vector Store (`yuedpao_coupons_e5`) & BM25 Index

In [ ]:
def index_coupons_to_chromadb(db_path: str = "../yuedpao_chatbot.db", chroma_path: str = "../data/chroma"):
    """
    อ่านคูปองจาก SQLite DB และแปลงเป็น Vector Passage เพิ่มเข้า ChromaDB (yuedpao_coupons_e5)
    """
    if not os.path.exists(db_path):
        print(f"⚠️ ไม่พบไฟล์ SQLite DB ที่ {db_path}")
        return None, None
        
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute("SELECT coupon_id, badge_title, badge_bg_color, discount_title, coupon_code, min_spend, expiry_date, valid_duration, detailed_condition, eligibility_tag FROM coupons")
    rows = cursor.fetchall()
    conn.close()
    
    documents = []
    doc_ids = []
    metadatas = []
    
    for r in rows:
        c_id, b_title, b_color, d_title, code, min_sp, exp, duration, cond, tag = r
        doc_text = (
            f"passage: คูปองส่วนลด YuedPao: {b_title} | โค้ดส่วนลด: {code} | หัวข้อส่วนลด: {d_title} | "
            f"ขั้นต่ำ: {min_sp} บาท | เงื่อนไขเพิ่มเติม: {cond} | ระยะเวลาใช้งาน: {duration} | สิทธิ์ผู้ใช้: {tag}"
        )
        documents.append(doc_text)
        doc_ids.append(c_id)
        metadatas.append({
            "coupon_id": c_id,
            "badge_title": b_title,
            "badge_bg_color": b_color,
            "discount_title": d_title,
            "coupon_code": code,
            "min_spend": min_sp,
            "expiry_date": exp,
            "valid_duration": duration,
            "detailed_condition": cond,
            "eligibility_tag": tag
        })
        
    print(f"📦 โหลดข้อมูลคูปองเตรียมทำ Vector Index จำนวน {len(documents)} รายการ...")
    
    # โหลด BERT Model
    model = SentenceTransformer('intfloat/multilingual-e5-small')
    embeddings = model.encode(documents, convert_to_tensor=False, show_progress_bar=False).tolist()
    
    os.makedirs(chroma_path, exist_ok=True)
    client = chromadb.PersistentClient(path=chroma_path)
    
    collection_name = "yuedpao_coupons_e5"
    if collection_name in [c.name for c in client.list_collections()]:
        client.delete_collection(collection_name)
        
    collection = client.create_collection(name=collection_name, metadata={"hnsw:space": "cosine"})
    collection.add(ids=doc_ids, documents=documents, embeddings=embeddings, metadatas=metadatas)
    
    # สร้าง BM25 Index
    bm25_corpus = [[t.strip().lower() for t in word_tokenize(doc.replace("passage: ", ""), engine="newmm") if t.strip()] for doc in documents]
    bm25_model = BM25Okapi(bm25_corpus)
    
    print(f"🚀 โหลดและสร้าง Vector Index ใน ChromaDB ('{collection_name}') & BM25 สำเร็จรวม {collection.count()} รายการ!")
    return collection, bm25_model

chroma_coupon_coll, coupon_bm25 = index_coupons_to_chromadb()

## 🔍 Step 5: ทดสอบ RRF Hybrid Search ค้นหาคูปองด้วยคำถามภาษาธรรมชาติ

In [ ]:
def search_coupons_rrf(query: str, collection, bm25_model, db_path: str = "../yuedpao_chatbot.db", top_k: int = 3) -> List[Dict[str, Any]]:
    """
    ทดสอบ RRF Hybrid Search (BM25 + ChromaDB Vector Search) สำหรับคูปอง
    """
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute("SELECT coupon_id, badge_title, badge_bg_color, discount_title, coupon_code, min_spend, expiry_date, valid_duration, detailed_condition, eligibility_tag FROM coupons")
    rows = cursor.fetchall()
    conn.close()
    
    if not rows or not collection or not bm25_model:
        return []
        
    doc_ids = [r[0] for r in rows]
    metadatas = [{
        "coupon_id": r[0], "badge_title": r[1], "badge_bg_color": r[2],
        "discount_title": r[3], "coupon_code": r[4], "min_spend": r[5],
        "expiry_date": r[6], "valid_duration": r[7], "detailed_condition": r[8], "eligibility_tag": r[9]
    } for r in rows]
    
    # 1. BM25 search
    q_tokens = [t.strip().lower() for t in word_tokenize(query, engine="newmm") if t.strip()]
    bm25_scores = bm25_model.get_scores(q_tokens)
    bm25_ranks = np.argsort(bm25_scores)[::-1]
    
    # 2. Vector search
    model = SentenceTransformer('intfloat/multilingual-e5-small')
    q_emb = model.encode(f"query: {query}", convert_to_tensor=False).tolist()
    vec_res = collection.query(query_embeddings=[q_emb], n_results=len(doc_ids))
    vec_rank_map = {doc_id: rank + 1 for rank, doc_id in enumerate(vec_res["ids"][0])}
    
    # 3. RRF Fusion
    scores = {}
    for bm25_rank, idx in enumerate(bm25_ranks):
        doc_id = doc_ids[idx]
        r_bm = bm25_rank + 1
        r_vec = vec_rank_map.get(doc_id, 9999)
        rrf_score = (1.0 / (60 + r_bm)) + (1.0 / (60 + r_vec))
        scores[doc_id] = {"score": rrf_score, "metadata": metadatas[idx]}
        
    sorted_res = sorted(scores.items(), key=lambda x: x[1]["score"], reverse=True)[:top_k]
    return [item[1]["metadata"] for item in sorted_res]

# ทดสอบค้นหาคูปองด้วยคำถามภาษาธรรมชาติ
test_queries = [
    "ขอคูปองส่วนลดสำหรับลูกค้าใหม่หน่อย",
    "มีโค้ดส่วนลด 100 บาทไหม",
    "อยากได้คูปองซื้อ 200 ขึ้นไป"
]

for tq in test_queries:
    print(f"\n🔍 Query: '{tq}'")
    results = search_coupons_rrf(tq, chroma_coupon_coll, coupon_bm25)
    for i, res in enumerate(results, 1):
        print(f"  {i}. โค้ด: {res['coupon_code']} | {res['discount_title']} | {res['detailed_condition']} | สิทธิ์: {res['eligibility_tag']}")

## 💬 Step 6: สร้าง LINE Flex Message Coupon Carousel (พร้อมนำไปใช้งานบน LINE Bot)

In [ ]:
def build_line_flex_coupon_carousel(coupons: List[Dict[str, Any]]) -> Dict[str, Any]:
    """
    สร้างโครงสร้าง LINE Flex Message (Carousel) จากข้อมูลคูปองตั๋ว
    เพื่อนำไปใช้ส่งการ์ดคูปองส่วนลดให้ผู้ใช้ใน LINE Chatbot
    """
    bubbles = []
    for c in coupons:
        bubble = {
            "type": "bubble",
            "size": "kilo",
            "header": {
                "type": "box",
                "layout": "vertical",
                "backgroundColor": c.get("badge_bg_color", "#EF4444"),
                "paddingAll": "15px",
                "contents": [
                    {
                        "type": "text",
                        "text": f"🎟️ {c.get('badge_title', 'ส่วนลดสินค้า')}",
                        "color": "#FFFFFF",
                        "weight": "bold",
                        "size": "sm",
                        "align": "center"
                    },
                    {
                        "type": "text",
                        "text": c["discount_title"],
                        "color": "#FFFFFF",
                        "weight": "bold",
                        "size": "xl",
                        "align": "center",
                        "margin": "md"
                    }
                ]
            },
            "body": {
                "type": "box",
                "layout": "vertical",
                "spacing": "md",
                "contents": [
                    {
                        "type": "box",
                        "layout": "horizontal",
                        "backgroundColor": "#F5F5F5",
                        "cornerRadius": "md",
                        "paddingAll": "8px",
                        "contents": [
                            {"type": "text", "text": "โค้ดส่วนลด:", "color": "#666666", "size": "xs", "flex": 3},
                            {"type": "text", "text": c.get("coupon_code", "-"), "color": "#D32F2F", "size": "xs", "weight": "bold", "align": "end", "flex": 5}
                        ]
                    },
                    {
                        "type": "box",
                        "layout": "baseline",
                        "spacing": "sm",
                        "contents": [
                            {"type": "text", "text": "เงื่อนไข:", "color": "#888888", "size": "xs", "flex": 2},
                            {"type": "text", "text": c.get("detailed_condition", "-"), "color": "#111111", "size": "xs", "weight": "bold", "flex": 5, "wrap": True}
                        ]
                    },
                    {
                        "type": "box",
                        "layout": "baseline",
                        "spacing": "sm",
                        "contents": [
                            {"type": "text", "text": "ระยะเวลา:", "color": "#888888", "size": "xs", "flex": 2},
                            {"type": "text", "text": c.get("valid_duration", "-") or "จนกว่าสิทธิ์จะหมด", "color": "#2E7D32", "size": "xs", "weight": "bold", "flex": 5, "wrap": True}
                        ]
                    }
                ]
            },
            "footer": {
                "type": "box",
                "layout": "vertical",
                "contents": [
                    {
                        "type": "button",
                        "action": {
                            "type": "clipboard",
                            "label": f"📋 คัดลอกโค้ด ({c.get('coupon_code', 'COUPON')})",
                            "clipboardText": c.get("coupon_code", "")
                        },
                        "style": "primary",
                        "color": "#EF4444",
                        "height": "sm"
                    }
                ]
            }
        }
        bubbles.append(bubble)
        
    return {
        "type": "flex",
        "altText": "🎟️ คูปองส่วนลดพิเศษ YuedPao พร้อมโค้ดลด",
        "contents": {
            "type": "carousel",
            "contents": bubbles
        }
    }

flex_message_json = build_line_flex_coupon_carousel(coupons_data)
print("✨ สร้าง LINE Flex Message Coupon Carousel พร้อม Promo Code เรียบร้อย!")
print(json.dumps(flex_message_json, ensure_ascii=False, indent=2)[:700] + "\n...")